# CropCop EAAI — Clean Kaggle Session Bootstrap\nThin orchestration only. The notebook establishes one global clock, retrieves secrets without printing them, clones the private repository with askpass, checks out the exact authorized commit, installs the frozen execution lock, validates the clean source, and delegates to repository code.\n

In [ ]:
import os, platform, shutil, stat, subprocess, sys, tempfile, time\nfrom pathlib import Path\nos.environ['PYTHONDONTWRITEBYTECODE'] = '1'\nos.environ['CROPCOP_NOTEBOOK_STARTED_MONOTONIC'] = repr(time.monotonic())\nos.environ.setdefault('CROPCOP_NOTEBOOK_HARD_LIMIT_SECONDS', str(12*3600))\nos.environ.setdefault('CROPCOP_NOTEBOOK_FINALIZATION_MARGIN_SECONDS', str(3600))\nassert platform.python_version() == '3.12.13', f'Python re-lock required: {platform.python_version()}'\ntry:\n    from kaggle_secrets import UserSecretsClient\n    _secrets = UserSecretsClient()\n    for _key in ('CROPCOP_GITHUB_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):\n        if not os.environ.get(_key):\n            os.environ[_key] = _secrets.get_secret(_key)\nexcept Exception as _exc:\n    raise RuntimeError('Required Kaggle Secrets could not be retrieved') from _exc\nsource_sha = os.environ.get('CROPCOP_SOURCE_GIT_COMMIT','').strip()\nlane = os.environ.get('CROPCOP_LANE','').strip()\nphase = os.environ.get('CROPCOP_EXECUTION_PHASE','smoke').strip()\nassert len(source_sha)==40 and lane in {'K1','K2','K3'} and phase in {'smoke','g1','calibration','principal'}\nrepo_url = os.environ.get('CROPCOP_REPOSITORY_URL','https://github.com/rana-m-ahmed/ResearchWork-CropCop.git')\nworkdir = Path(os.environ.get('CROPCOP_REPO_WORKDIR','/kaggle/working/cropcop-je')).resolve()\nif workdir.exists(): shutil.rmtree(workdir)\nwith tempfile.TemporaryDirectory() as td:\n    askpass = Path(td)/'askpass.py'\n    askpass.write_text("#!/usr/bin/env python3\\nimport os,sys\\np=sys.argv[1] if len(sys.argv)>1 else ''\\nprint('x-access-token' if 'Username' in p else os.environ['CROPCOP_GITHUB_TOKEN'])\\n")\n    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)\n    env=dict(os.environ); env['GIT_ASKPASS']=str(askpass); env['GIT_TERMINAL_PROMPT']='0'\n    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',repo_url,str(workdir)],env=env,check=True)\nsubprocess.run(['git','-C',str(workdir),'checkout','--detach',source_sha],check=True)\nactual=subprocess.check_output(['git','-C',str(workdir),'rev-parse','HEAD'],text=True).strip()\nassert actual==source_sha\nlockfile=workdir/'journal_extension/requirements-training.lock.txt'\nsubprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-input','-r',str(lockfile)],check=True)\nbootstrap_out=Path(os.environ['CROPCOP_OUTPUT_ROOT'])/'bootstrap'/'clean_session.json'\nsubprocess.run([sys.executable,str(workdir/'journal_extension/kaggle/bootstrap_clean_session.py'),'--repo-root',str(workdir),'--authorized-source-sha',source_sha,'--output',str(bootstrap_out)],cwd=workdir,check=True)\nif phase == 'smoke':\n    smoke_out = Path(os.environ['CROPCOP_OUTPUT_ROOT'])/'smoke'\n    subprocess.run([sys.executable,str(workdir/'journal_extension/scripts/smoke_infrastructure.py'),'--repo-root',str(workdir),'--authorized-source-sha',source_sha,'--synthetic-bundle-dir',os.environ['CROPCOP_SYNTHETIC_SMOKE_BUNDLE'],'--output-dir',str(smoke_out),'--durable-store-kind',os.environ['CROPCOP_SMOKE_DURABLE_STORE_KIND'],'--durable-store-locator',os.environ['CROPCOP_SMOKE_DURABLE_STORE_LOCATOR']],cwd=workdir,check=True)\nelif phase == 'g1':\n    subprocess.run([sys.executable,str(workdir/'journal_extension/kaggle/run_g1.py')],cwd=workdir,check=True)\nelse:\n    subprocess.run([sys.executable,str(workdir/'journal_extension/kaggle/run_lane.py'),'--lane',lane,'--phase',phase],cwd=workdir,check=True)\n